# Initialize and import packages

In [25]:
import geopandas as gpd
import pandas as pd
import math
import ee
import geemap
import os
import json
import ast

In [3]:
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='ee-curuai2')

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
wrk_directory = r"C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Parameters Time series\Water Period Definitions"

# Import image collections

Landsat 5 and Landsat 7 - PY6S

In [6]:
landsat5 = ee.ImageCollection("projects/ee-curuai2/assets/Py6S/LD5/Landsat5")\
            .merge(ee.ImageCollection("projects/ee-curuai/assets/Py6S/LD7/Landsat7"))\
            .merge(ee.ImageCollection('projects/ee-curuai2/assets/Py6S/LD7/Landsat7'))\
            .select([ 'B1', 'B2', 'B3', 'B4', 'B5', 'B7'])
print(ee.Date(landsat5.sort('system:time_start',True).first().get('system:time_start')).format().getInfo())
print(ee.Date(landsat5.sort('system:time_start',False).first().get('system:time_start')).format().getInfo())
print(landsat5.size().getInfo())

1985-02-07T13:24:25
2024-01-16T11:16:49
2446


Landsat 8 and LANDSAT 9 -  PY6S

In [7]:
landsat8 = (ee.ImageCollection("projects/ee-curuai/assets/Py6S/LD8/Landsat8")
            .merge(ee.ImageCollection('projects/ee-curuai2/assets/Py6S/LD9/Landsat9'))
            .select(['B2', 'B3', 'B4', 'B5', 'B6', 'B7']))
print(ee.Date(landsat8.sort('system:time_start',True).first().get('system:time_start')).format().getInfo())
print(ee.Date(landsat8.sort('system:time_start',False).first().get('system:time_start')).format().getInfo())
print(landsat8.size().getInfo())

2013-05-11T13:55:54
2025-12-15T13:48:04
1081


# Transform into remote sensing reflectance and sunglint correction

In [8]:
def deglint(img):
  '''Divide corrected image by pi.
  Rrs_sat_ac = Rsat_ac / pi
  Perform deglint correction:
  Rrs_sat_ac_deglint(VNIR) = Rrs_sat_ac(VNIR) - Rrs_sat_ac(SWIR)
  Correction following INPE CURUAI publication.
  '''
  Rrs = img.divide(math.pi)
  deglint = Rrs.select(['blue_mean','green_mean','red_mean','nir_mean','swir1','swir2']) \
    .subtract(Rrs.select('swir1'))

  return (deglint
          .copyProperties(img,['system:time_start','CLOUD_COVER',"system:index"]))

## Standardize band names

In [9]:
name_bands = ['blue_mean','green_mean','red_mean','nir_mean','swir1','swir2']

Landsat 5 and 7

In [10]:
# rename bands
ld5 = landsat5.map(lambda img: img.rename(name_bands))
display(ld5.size().getInfo())

2446

Landsat 8 and 9

In [11]:
ld8 = landsat8.map(lambda img: img.rename(name_bands))
display(ld8.size().getInfo())

1081

In [12]:
merge_col = ld5.merge(ld8).sort('system:time_start').map(deglint)
merge_col.limit(5)

In [13]:
merge_col.size()

# Import period dates based on Óbidos water level

In [14]:
# Import period dates
df_period_limits = pd.read_csv(os.path.join(wrk_directory, 'water_period_limits.csv')).drop(columns=['Unnamed: 0'])
# df_period_limits
# # Convert to datetime objects
df_period_limits['lim_next'] = pd.to_datetime(df_period_limits['lim_next'])
df_period_limits['lim_previous'] = pd.to_datetime(df_period_limits['lim_previous'])

# # # Sort by date to ensure chronological order
df_period_limits = df_period_limits.sort_values(by='lim_next').reset_index(drop=True)
# # Fills 'lim_next' with 'lim_previous' + 3 months ONLY where 'lim_next' is currently missing
df_period_limits['lim_next'] = df_period_limits['lim_next'].fillna(
    df_period_limits['lim_previous'] + pd.DateOffset(months=3)
)

df_period_limits['lim_previous'] = df_period_limits['lim_previous'].fillna(
    df_period_limits['lim_next'] + pd.DateOffset(months=-3)
)

# # Verify the order
display(df_period_limits.tail())


,lim_previous,lim_next,water_period
220,2024-03-16 12:00:00,2024-06-08 06:00:00,HW
221,2024-06-08 06:00:00,2024-08-23 18:00:00,F
222,2024-08-23 18:00:00,2024-11-23 00:00:00,LW
223,2024-11-23 00:00:00,2025-03-09 00:00:00,R
224,2025-03-09 00:00:00,2025-06-09 00:00:00,HW


In [15]:
fill_periods = {'lim_previous':['2025-06-09 00:00:00','2025-09-09 00:00:00','2025-12-09 00:00:00'],
                "lim_next":['2025-09-09 00:00:00','2025-12-09 00:00:00','2026-01-01 00:00:00'],
                'water_period':['F','LW',"R"]}
fill_periods = pd.DataFrame(fill_periods)

df_period_limits = pd.concat([df_period_limits, fill_periods]).reset_index(drop=True)

# Coerce the entire mixed column back to datetime
df_period_limits['lim_previous'] = pd.to_datetime(df_period_limits['lim_previous'])
df_period_limits['lim_next'] = pd.to_datetime(df_period_limits['lim_next'])

# Now .dt works
df_period_limits['year'] = df_period_limits['lim_previous'].dt.year
df_period_limits['month'] = df_period_limits['lim_previous'].dt.month
df_period_limits

,lim_previous,lim_next,water_period,year,month
0,1968-06-30 12:00:00,1968-09-30 12:00:00,HW,1968,6
1,1968-09-30 12:00:00,1969-08-01 12:00:00,F,1968,9
2,1969-08-01 12:00:00,1970-02-07 18:00:00,LW,1969,8
3,1970-02-07 18:00:00,1970-04-24 06:00:00,R,1970,2
4,1970-04-24 06:00:00,1970-07-16 18:00:00,HW,1970,4
...,...,...,...,...,...
223,2024-11-23 00:00:00,2025-03-09 00:00:00,R,2024,11
224,2025-03-09 00:00:00,2025-06-09 00:00:00,HW,2025,3
225,2025-06-09 00:00:00,2025-09-09 00:00:00,F,2025,6
226,2025-09-09 00:00:00,2025-12-09 00:00:00,LW,2025,9


# Create mosaics defined by water level in Óbidos

In [16]:
list_images = []

# Use the sorted dataframe
for i in range(0, len(df_period_limits)):
    
    # Format dates as strings for GEE
    start_date = df_period_limits['lim_previous'][i].strftime('%Y-%m-%d')
    end_date = df_period_limits['lim_next'][i].strftime('%Y-%m-%d')

    filter_dates = merge_col.filterDate(ee.Date(start_date), ee.Date(end_date))

    # Check if images exist in this range
    # using size() is often safer/faster than aggregate_count for simple checks
    if filter_dates.size().getInfo() > 0:
        image = filter_dates.median()

        if image.bandNames().size().getInfo() > 0:
            list_images.append(image
                .set('year', str(df_period_limits['year'][i]))
                .set('month_init', str(df_period_limits['month'][i]))
                .set('month_end', str(df_period_limits['month'][i]))
                .set('system:time_start', ee.Date(start_date).millis()) # Use millis for system:time_start
                .set('time_start', start_date)
                .set('time_finish', end_date)
                .set('band_count', image.bandNames().length())
            )
    else:
        continue



In [17]:
# Convert list to ImageCollection
period_mosaic = ee.ImageCollection(list_images)

print(f"Created mosaic with {period_mosaic.size().getInfo()} images")

Created mosaic with 163 images


# Calculate area

In [18]:
# Mask land function
def hsvComposite (image):
    composite = image.select(['blue_mean','green_mean','red_mean']).rgbToHsv()#.clip(limits);
    hue = composite.select("hue");
    max_mask = hue.lte(0.9)
    min_mask = hue.gte(0.3)
    return image.updateMask(max_mask).updateMask(min_mask).select(['blue_mean','green_mean','red_mean','nir_mean']);

In [19]:
def area_calc(img):
  '''receives an image of curuai and returns water surface area in km2 within the floodplain limits
  as a property of the input image'''

  # Get a pixel area image.
  pixel_area = ee.Image.pixelArea()

  floodplain = ee.FeatureCollection('projects/ee-curuai2/assets/varzea_alagavel')
  image = hsvComposite(img)
  img_mask = image.gt(0)

  areaImage = img_mask.multiply(pixel_area)

  area = areaImage.reduceRegion(**{
    'reducer': ee.Reducer.sum(),
    'geometry': floodplain.geometry(),
    'scale': 30,
    'maxPixels': 1e10
    })
  return image.set('area_km2',ee.Number(area.get('red_mean')).divide(1e6))


In [20]:
#calculate area and apply for each mosaic image
period_area = period_mosaic.map(area_calc)

period_area.limit(3)

# Apply Model and Classify Images

In [27]:
data_directory = r'C:\Users\l_v_v\Documents\GitHub\time_series_curuai\datasets\Landsat Sampling\Merged Landsat Data'
model_directory ="C:/Users/l_v_v/Documents/GitHub/time_series_curuai/datasets/Parameters Time series/TSS Modeling"

In [28]:
#import selected model
model_info = pd.read_csv(os.path.join(model_directory,'model_selection.csv')).iloc[0]
params = ast.literal_eval(model_info['Params'])
params['seed'] = 123
params

{'numberOfTrees': 500,
 'shrinkage': None,
 'samplingRate': 0.6,
 'loss': 'Huber',
 'seed': 123}

In [29]:
# import training data as dataframe
df_data = pd.read_csv(os.path.join(data_directory,'min_date.csv'))
df_data = df_data.drop(columns=['Unnamed: 0', 'CHLOROPHYLL',
                      'CHLOROPHYLL_B', 'DOC', 'dif_date_point',
                      'N_TOTAL', 'N_TOTAL_DISSOLVED',
                      'POC', 'P_ORGANIC', 'P_TOTAL',
                      'SILICA',  'TOC', 'duplicated'],axis=1).rename(columns={'SPM':"TSS"})
# filter parameters that will be used in the models
df_subset = df_data[['TSS','blue_mean',
       'green_mean',
       'nir_mean',
       'red_mean',
       'datetime',
       'WATER_PERIOD',
       'LOCATION',
       'LONGITUDE',
       'LATITUDE']].copy()
# remove empty values
df_subset = df_subset.dropna()
df_subset.isna().sum()
# obtain date from datetime
df_subset['date'] = df_subset['datetime'].apply(lambda row: row[:10])
#transform dataframe in a geodataframe (geometry column with point location)
gdf = gpd.GeoDataFrame(
    df_subset, geometry=gpd.points_from_xy(df_subset.LONGITUDE, df_subset.LATITUDE),
    crs="EPSG:4326"
)
##Convert geodataframe to json - necessary to be read in GEE
dataset_json = gdf.to_json()
#load and select the features of the json data
reg_data = json.loads(dataset_json)
reg_data = reg_data['features']
##transform json in in gee object = feature collection
reg_data = ee.FeatureCollection(reg_data)
print(reg_data.size().getInfo())

202


In [30]:
predictors = ['blue_mean', 'green_mean','red_mean', 'nir_mean']

In [31]:
# Train a GBR regressor with previously defined parameters.
trained = (ee.Classifier.smileGradientTreeBoost(**params)
           .train(features = reg_data, 
                  classProperty = "TSS", 
                  inputProperties=predictors)
           .setOutputMode('REGRESSION'))

## by elevation

In [34]:
spm_period_classified = period_area.select(predictors).map(lambda img: img.addBands(img.classify(classifier=trained).rename("TSS")))

display(spm_period_classified.size().getInfo())

163

In [35]:
spm_period_classified.limit(5)

# Export as Asset: Image Collection

In [36]:
prj = landsat8.first().select('B4').projection().getInfo()
scale = landsat8.first().select('B4').projection().nominalScale().getInfo()
region = ee.FeatureCollection(ee.List([ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/lim_varzea').geometry().buffer(30)),ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/points_curuai').geometry().bounds().buffer(30))])).geometry().bounds()

## by water level

In [37]:
def export_img(img):
      # define YOUR assetID
    # export

    fname = ee.String(img.get('time_start')).getInfo()
    export = ee.batch.Export.image.toAsset(\
        image=ee.Image(img),
        description= 'ld_mosaic_'+fname,
        assetId = 'projects/ee-curuai2/assets/landsat_water_period/water_period/mosaic_'+fname,#change the properties to add in images here
        region = region,
        crs = prj['crs'],
        scale = scale,
        maxPixels = 1e13)

    # # uncomment to run the export
    export.start()
    print('exporting ' +fname + '--->done')

    # return img



In [ ]:
# geemap.ee_export_image_collection_to_asset(spm_period_classified, scale=30,crs='EPSG:32621',maxPixels=1e13,)

In [ ]:
col_length = spm_period_classified.size().getInfo()
# print(col_length)
# cannot map the function because the function runs on both client and server sides, so we need to use a loop
# for very big time series it is recommended to break the series and export data in parts
# not only because of the loop but also because of how exporting to assets works in Google Earth Engine
# and you can run into problems if too much information is exported at the same time
lista = spm_period_classified.toList(col_length)
for i in range(0,col_length):
    # print(i)
    # lista = x.toList(50)
    img = ee.Image(lista.get(i))
    # print(img.getInfo())
    export_img(img)

# Visualize gif

In [ ]:
# import collection to generate gif
colecao = ee.ImageCollection('projects/ee-curuai2/assets/landsat_water_period/water_period_discharge')
colecao.first().bandNames()

In [ ]:
colecao.size()

In [ ]:
# Define arguments for animation function parameters.
video_args = {
    "dimensions": 600,
    "region": ee.FeatureCollection('projects/ee-curuai2/assets/lim_varzea').geometry(),
    "framesPerSecond": 5,
    "bands": ["classification"],
    "min": 1,
    "max": 350,
    "palette": ['blue','green', 'yellow', 'orange','red'],
}

In [ ]:
# geemap.download_ee_video(spm_classified, video_args, '/content/classification_SPM.gif')
geemap.download_ee_video(colecao, video_args, 'classification_SPM_dis.gif')

In [ ]:
geemap.add_text_to_gif(
    '/content/classification_SPM_dis.gif',
    '/content/classification_SPM_text_dis.gif',
    xy=("3%", "5%"),
    text_sequence=colecao.aggregate_array('time_start').getInfo(),
    font_size=30,
    font_color="#ffffff",
    add_progress_bar=True,
)

In [ ]:
# Define arguments for animation function parameters.
video_args = {
    "dimensions": 600,
    "region": ee.FeatureCollection(ee.List([ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/lim_varzea').geometry().buffer(30)),ee.Feature(ee.FeatureCollection('projects/ee-curuai2/assets/points_curuai').geometry().bounds().buffer(30))])).geometry().bounds(),
    "framesPerSecond": 5,
    "bands": ["red_mean",'green_mean','blue_mean'],
    "min": 0,
    "max": 0.05,
}

In [ ]:
geemap.download_ee_video(colecao, video_args, '/content/water_period.gif')

In [ ]:
geemap.add_text_to_gif(
    '/content/water_period.gif',
    '/content/water_period_text.gif',
    xy=("3%", "5%"),
    text_sequence=colecao.aggregate_array('time_start').getInfo(),
    font_size=30,
    font_color="#ffffff",
    add_progress_bar=True,
)